In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
#from langchain_classic.chains.retrieval import create_retrieval_chain
#from langchain_classic.chains.combine_documents import create_stuff_documents_chain


C:\Users\Abhinesh Singh\AppData\Local\Temp\ipykernel_4976\3079736742.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\RAG\RAG\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Load the txt file
loader = TextLoader("langchain_sample.txt")
raw_docs = loader.load()

# Split text into document chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
docs = splitter.split_documents(raw_docs)
docs



[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.'),
 Document(metadata={

In [3]:
 ## user Query 
query = "How can i use langchain to build an application with memory and tools?"

In [4]:
### FAISS use for vectorStore  and HuggingFace embeddings model 

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(model_name ="all-MiniLM-L6-v2")
vectorStore = FAISS.from_documents(docs,embedding_model)
retriever = vectorStore.as_retriever(search_kwargs={"k":8})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10263.79it/s]


In [5]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002A577AD22D0>, search_kwargs={'k': 8})

In [6]:
## Prompt  and use the LLM
import os
from dotenv import load_dotenv

# Load .env first
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
# Check key
print(os.getenv("GROQ_API_KEY"))



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


In [8]:

from langchain.chat_models import init_chat_model
llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

In [9]:
# Prompt Template

prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question "{question}"

Documents:
{documents}   

Instruction:
-Think about the relevance of each document to the user's question.
-Return a list of document indices in ranked order, starting from the most relevant.

Output Format: comma- separated document indices (e.g., 2,1,3,0,...)                                   
""")

In [10]:
retrieved_docs = retriever.invoke(query)
retrieved_docs

[Document(id='a05db6a8-f56c-480f-8d3c-59e95f9018c0', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='cd042c7e-fa3e-4721-8d6f-2de476d10023', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(id='4878a02e-9399-4d21-b48d-11d9efc249a4', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond mo

In [11]:
chain = prompt| llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user\'s question.\n\nUser Question "{question}"\n\nDocuments:\n{documents}   \n\nInstruction:\n-Think about the relevance of each document to the user\'s question.\n-Return a list of document indices in ranked order, starting from the most relevant.\n\nOutput Format: comma- separated document indices (e.g., 2,1,3,0,...)                                   \n')
| ChatGroq(metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.8'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False

In [12]:
doc_line = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_line)
doc_line

['1. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.',
 '2. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.',
 '3. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.',
 '4. Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.',
 '5. Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-s

In [13]:
response = chain.invoke({"question":query,"documents":formatted_docs})
response

'To answer the user\'s question about using LangChain to build an application with memory and tools, we need to rank the provided documents based on their relevance to this specific query. \n\n1. **Document 4** is the most relevant because it directly talks about "memory" in LangChain, which is a key component of the user\'s question. Memory enables context retention across multiple steps in a conversation or task.\n\n2. **Document 3** is the next most relevant as it discusses tool integration, which is another crucial aspect of the user\'s question. Tool integration allows LLMs to interact with external systems.\n\n3. **Document 1** provides a general overview of LangChain, including its components for prompt management, chains, memory, and agents. It\'s relevant but broader than documents 4 and 3.\n\n4. **Document 5** talks about agents in LangChain, which use LLMs to decide which tools to use. This is relevant to building an application with tools but is more specific to multi-step 

In [15]:
# Parse and rerank 
indices = [int(x.strip()) - 1 for x in response.split(",") if x.strip().isdigit()]
indices

[2, 0, 4, 5, 1, 6]

In [16]:
reranked_docs = [retrieved_docs[i] for i in indices if 0 <= i <len(retrieved_docs)]
reranked_docs

[Document(id='4878a02e-9399-4d21-b48d-11d9efc249a4', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.'),
 Document(id='a05db6a8-f56c-480f-8d3c-59e95f9018c0', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='36e08b7d-bf38-4dd2-9a90-7d029ee94c08', metadata={'source': 'langchain_sample.txt'}, page_content='Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks like question answering with search and code execution.'),
 Document(id='3a94d3ed-ca2

In [17]:
# show result 
print(f"\n Final Reranked Result:")
for i, doc in enumerate(reranked_docs,1):
    print(f"\nRank {i}:\n{doc.page_content}")


 Final Reranked Result:

Rank 1:
LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.

Rank 2:
LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.

Rank 3:
Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks like question answering with search and code execution.

Rank 4:
Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.

Rank 5:
LangChain integrates with many third-party services 

# RAG with Re-Ranking Pipeline

```mermaid
flowchart TD
    A[TXT File] --> B[Load Documents]
    B --> C[Chunking]
    C --> D[Create Embeddings]
    D --> E[Store in FAISS Vector Store]
    E --> F[User Query]
    F --> G[Query Embedding]
    G --> H[FAISS Similarity Search]
    H --> I[Retrieve Top 8 Documents]
    I --> J[LLM Re-Ranking]
    J --> K[Reorder Documents by Relevance]
    K --> L[Best Ranked Documents]
```